# RAG-IDEArq — Normalización de Autores v2 (Fuzzy Matching)

Segunda pasada de deduplicación sobre el grafo `dedup-v1`. Usa similitud
combinada (Jaccard + Levenshtein) para fusionar autores con nombres muy
similares pero slugs distintos.

**Casos que resuelve**:
- Iniciales vs nombre completo: `António M. Monge Soares` vs `António Monge Soares`
- Apellidos extra: `Pablo Arias` vs `Pablo Arias Cabal`
- Múltiples formas: `F. Xavier Oms` / `Oms, Xavier` / `Xavier Oms Arias`
- Puntos y guiones: `Manuel González-Morales` vs `Manuel R. González Morales`

**Algoritmo**: similitud = max(Jaccard(tokens), 1-Levenshtein/max_len)
- ≥ 0.85 → fusión automática
- 0.70-0.85 → caso dudoso para revisión manual
- < 0.70 → no se fusiona

In [1]:
import os, sys, re, unicodedata
from pathlib import Path
from collections import Counter, defaultdict

try:
    PROJECT_ROOT = Path.cwd().parent.parent
except NameError:
    PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env", override=True)

from rdflib import Graph, Namespace, URIRef, Literal, RDF
from rdflib.namespace import FOAF, XSD

BIBO = Namespace("http://purl.org/ontology/bibo/")
DC_  = Namespace("http://purl.org/dc/terms/")
SKOS = Namespace("http://www.w3.org/2004/02/skos/core#")
IDEARQ = Namespace("http://idearq.org/resource/")

DATA_DIR = PROJECT_ROOT / "data" / "biblio-graph"
INPUT_NT = DATA_DIR / "idearq-graph-instances.nt"  # ← lee el original
OUTPUT_NT = DATA_DIR / "idearq-graph-instances-dedup-v2.nt"
OUTPUT_TTL = DATA_DIR / "idearq-graph-instances-dedup-v2.ttl"

print(f"Project root: {PROJECT_ROOT}")
print(f"Input NT:     {INPUT_NT}")
print(f"Output NT:    {OUTPUT_NT}")
print(f"Output TTL:   {OUTPUT_TTL}")


Project root: /home/raglinux/RAG
Input NT:     /home/raglinux/RAG/data/biblio-graph/idearq-graph-instances.nt
Output NT:    /home/raglinux/RAG/data/biblio-graph/idearq-graph-instances-dedup-v2.nt
Output TTL:   /home/raglinux/RAG/data/biblio-graph/idearq-graph-instances-dedup-v2.ttl


In [2]:
print(f"Cargando {INPUT_NT} ...")
g = Graph()
g.parse(str(INPUT_NT), format="nt")
print(f"  {len(g):,} triples cargados")

# Extraer autores: URI → nombre + nº artículos
author_data = {}  # uri → {name, n_articles}
for s, _, _ in g.triples((None, FOAF.name, None)):
    uri = str(s)
    name = str(list(g.objects(s, FOAF.name))[0])
    n_articles = len(list(g.subjects(DC_.creator, s)))
    author_data[uri] = {"name": name, "n_articles": n_articles}

print(f"  {len(author_data)} URIs únicas de autor en grafo v1")

names = [d["name"] for d in author_data.values()]
print(f"  {len(set(names))} nombres distintos")

Cargando /home/raglinux/RAG/data/biblio-graph/idearq-graph-instances.nt ...


  17,306 triples cargados
  2794 URIs únicas de autor en grafo v1
  2794 nombres distintos


In [3]:
# ── PASO 1: Normalización por slug exacto (como v1) ────────────────
def slugify(name: str) -> str:
    """Slug para deduplicación exacta (v1)."""
    name = name.strip()
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("ascii")
    name = name.lower()
    name = name.replace(",", " ").replace(".", " ").replace("-", " ")
    name = re.sub(r"[^a-z0-9 ]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    tokens = sorted(name.split())
    return " ".join(tokens)

# ── PASO 2: Normalización + fuzzy matching (v2) ────────────────────
def normalize_name(name: str) -> str:
    """Normalizar nombre para comparación fuzzy."""
    name = name.strip()
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("ascii")
    name = name.lower()
    name = name.replace(",", " ").replace(".", " ").replace("-", " ")
    name = re.sub(r"[^a-z0-9 ]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name

def jaccard_similarity(s1: str, s2: str) -> float:
    """Similitud Jaccard sobre conjuntos de tokens."""
    t1 = set(s1.split())
    t2 = set(s2.split())
    if not t1 or not t2:
        return 0.0
    return len(t1 & t2) / len(t1 | t2)

def levenshtein_distance(s1: str, s2: str) -> int:
    """Distancia de Levenshtein (DP)."""
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)
    if len(s2) == 0:
        return len(s1)
    prev_row = list(range(len(s2) + 1))
    for i, c1 in enumerate(s1):
        curr_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = prev_row[j + 1] + 1
            deletions = curr_row[j] + 1
            substitutions = prev_row[j] + (c1 != c2)
            curr_row.append(min(insertions, deletions, substitutions))
        prev_row = curr_row
    return prev_row[-1]

def normalized_levenshtein(s1: str, s2: str) -> float:
    """Similitud Levenshtein normalizada [0,1]."""
    if not s1 or not s2:
        return 0.0
    dist = levenshtein_distance(s1, s2)
    return 1 - dist / max(len(s1), len(s2))

def combined_similarity(name1: str, name2: str) -> float:
    """Similitud combinada: max(Jaccard, Levenshtein)."""
    n1 = normalize_name(name1)
    n2 = normalize_name(name2)
    if n1 == n2:
        return 1.0
    jaccard = jaccard_similarity(n1, n2)
    levenshtein = normalized_levenshtein(n1, n2)
    return max(jaccard, levenshtein)

# Pruebas
tests = [
    ("António M. Monge Soares", "António Monge Soares"),
    ("Pablo Arias", "Pablo Arias Cabal"),
    ("F. Xavier Oms Arias", "Oms, Xavier F."),
    ("Manuel González-Morales", "Manuel R. González Morales"),
    ("G. J. VAN KLINKEN", "Gibaja, Juan F."),
]
print("Pruebas de similitud:")
for a, b in tests:
    sim = combined_similarity(a, b)
    print(f"  {a:35s} ≈ {b:35s} → {sim:.3f}")

# Slugify test
print("\nPruebas de slugify (dedup exacta):")
for a, b in tests[:3]:
    print(f"  {a:35s} → {slugify(a)}")
    print(f"  {b:35s} → {slugify(b)}")


Pruebas de similitud:
  António M. Monge Soares             ≈ António Monge Soares                → 0.909
  Pablo Arias                         ≈ Pablo Arias Cabal                   → 0.667
  F. Xavier Oms Arias                 ≈ Oms, Xavier F.                      → 0.750
  Manuel González-Morales             ≈ Manuel R. González Morales          → 0.920
  G. J. VAN KLINKEN                   ≈ Gibaja, Juan F.                     → 0.267

Pruebas de slugify (dedup exacta):
  António M. Monge Soares             → antonio m monge soares
  António Monge Soares                → antonio monge soares
  Pablo Arias                         → arias pablo
  Pablo Arias Cabal                   → arias cabal pablo
  F. Xavier Oms Arias                 → arias f oms xavier
  Oms, Xavier F.                      → f oms xavier


In [4]:
THRESHOLD_MERGE = 0.85
THRESHOLD_REVIEW = 0.70

# ── PASO 1: Deduplicación por slug exacto ──────────────────────────
uri_to_slug = {uri: slugify(d["name"]) for uri, d in author_data.items()}
slug_to_uris = defaultdict(set)
for uri, slug in uri_to_slug.items():
    slug_to_uris[slug].add(uri)

v1_clusters = {slug: uris for slug, uris in slug_to_uris.items() if len(uris) > 1}
v1_merged = sum(len(uris) - 1 for uris in v1_clusters.values())
print(f"Paso 1 (slug exacto): {len(v1_clusters)} grupos, {v1_merged} URIs fusionadas")

# ── PASO 2: Fuzzy matching sobre los slugs ya agrupados ────────────
# Para cada slug, elegir el nombre canónico (más artículos)
slug_to_canonical_name = {}
for slug, uris in slug_to_uris.items():
    best_uri = max(uris, key=lambda u: author_data[u]["n_articles"])
    slug_to_canonical_name[slug] = author_data[best_uri]["name"]

# Ahora fuzzy matching entre slugs distintos
unique_slugs = list(slug_to_uris.keys())
clusters = []  # [{slugs: [], uris: [], max_sim: float}]
doubtful_pairs = []

for i, slug1 in enumerate(unique_slugs):
    name1 = slug_to_canonical_name[slug1]
    best_sim = 0.0
    best_cluster = None
    
    for j, slug2 in enumerate(unique_slugs[:i]):
        name2 = slug_to_canonical_name[slug2]
        sim = combined_similarity(name1, name2)
        
        if sim >= THRESHOLD_MERGE:
            for cluster in clusters:
                if slug2 in cluster["slugs"]:
                    if sim > best_sim:
                        best_sim = sim
                        best_cluster = cluster
                    break
        elif sim >= THRESHOLD_REVIEW:
            doubtful_pairs.append((name1, name2, sim))
    
    if best_cluster is not None:
        best_cluster["slugs"].add(slug1)
        best_cluster["uris"].update(slug_to_uris[slug1])
        best_cluster["max_sim"] = max(best_cluster["max_sim"], best_sim)
    else:
        clusters.append({
            "slugs": {slug1},
            "uris": set(slug_to_uris[slug1]),
            "max_sim": 1.0,
        })

merge_clusters = [c for c in clusters if len(c["slugs"]) > 1]

print(f"Paso 2 (fuzzy ≥{THRESHOLD_MERGE}): {len(merge_clusters)} grupos adicionales")
print(f"Pares dudosos ({THRESHOLD_REVIEW}-{THRESHOLD_MERGE}): {len(doubtful_pairs)}")

total_fused = sum(len(c["uris"]) - 1 for c in merge_clusters)
print(f"URIs adicionales fusionadas por fuzzy: {total_fused}")
print(f"\nTotal fusionado (v1 + v2): {v1_merged + total_fused} URIs")


Paso 1 (slug exacto): 288 grupos, 317 URIs fusionadas


Paso 2 (fuzzy ≥0.85): 66 grupos adicionales
Pares dudosos (0.7-0.85): 236
URIs adicionales fusionadas por fuzzy: 94

Total fusionado (v1 + v2): 411 URIs


In [5]:
# Reporte de fusiones por slug exacto (Paso 1)
print(f"{'='*70}")
print(f"  PASO 1: FUSIONES POR SLUG EXACTO ({len(v1_clusters)} grupos)")
print(f"{'='*70}")
for i, (slug, uris) in enumerate(sorted(v1_clusters.items(), key=lambda x: -len(x[1]))[:10]):
    best_uri = max(uris, key=lambda u: author_data[u]["n_articles"])
    canonical = author_data[best_uri]["name"]
    total = sum(author_data[u]["n_articles"] for u in uris)
    print(f"\n  Grupo {i+1} ({len(uris)} URIs, {total} artículos)")
    print(f"    Slug: {slug}")
    print(f"    Canónico: {canonical}")
    for uri in sorted(uris):
        d = author_data[uri]
        marker = " ★" if uri == best_uri else ""
        print(f"      {d['n_articles']:>3}  {d['name']}{marker}")

# Reporte de fusiones fuzzy (Paso 2)
print(f"\n\n{'='*70}")
print(f"  PASO 2: FUSIONES POR FUZZY MATCHING (≥{THRESHOLD_MERGE})")
print(f"{'='*70}")
for i, cluster in enumerate(sorted(merge_clusters, key=lambda c: -len(c["uris"]))):
    best_slug = max(cluster["slugs"], key=lambda s: sum(author_data[u]["n_articles"] for u in slug_to_uris[s]))
    canonical_name = slug_to_canonical_name[best_slug]
    total_articles = sum(author_data[u]["n_articles"] for u in cluster["uris"])
    
    print(f"\n  Cluster {i+1} ({len(cluster['uris'])} URIs, {total_articles} artículos, sim={cluster['max_sim']:.3f})")
    print(f"    Canónico: {canonical_name}")
    for slug in sorted(cluster["slugs"]):
        for uri in sorted(slug_to_uris[slug]):
            d = author_data[uri]
            marker = " ★" if d["name"] == canonical_name else ""
            print(f"      {d['n_articles']:>3}  {d['name']}{marker}")

print(f"\n\n{'='*70}")
print(f"  CASOS DUDOSOS ({THRESHOLD_REVIEW} ≤ sim < {THRESHOLD_MERGE})")
print(f"{'='*70}")
for name1, name2, sim in sorted(doubtful_pairs, key=lambda x: -x[2])[:30]:
    a1 = sum(author_data[u]["n_articles"] for u in slug_to_uris[slugify(name1)])
    a2 = sum(author_data[u]["n_articles"] for u in slug_to_uris[slugify(name2)])
    print(f"  [{sim:.3f}] {name1} ({a1}) ≈ {name2} ({a2})")

if len(doubtful_pairs) > 30:
    print(f"  ... y {len(doubtful_pairs) - 30} más")


  PASO 1: FUSIONES POR SLUG EXACTO (288 grupos)

  Grupo 1 (3 URIs, 14 artículos)
    Slug: guillem jorda perez
    Canónico: Pérez Jordà, Guillem
        4  Guillem Pérez Jordà
        1  Jordà, Guillem Pérez
        9  Pérez Jordà, Guillem ★

  Grupo 2 (3 URIs, 12 artículos)
    Slug: garcia leonardo sanjuan
    Canónico: Leonardo García Sanjuán
        2  García Sanjuán, Leonardo
        9  Leonardo García Sanjuán ★
        1  Sanjuán, Leonardo García

  Grupo 3 (3 URIs, 3 artículos)
    Slug: maicas ramos ruth
    Canónico: Ramos, Ruth Maicas
        1  Maicas Ramos, Ruth
        1  Ramos, Ruth Maicas ★
        1  Ruth Maicas Ramos

  Grupo 4 (3 URIs, 4 artículos)
    Slug: norberto palomares zumajo
    Canónico: Zumajo, Norberto Palomares
        1  Norberto Palomares Zumajo
        1  Palomares Zumajo, Norberto
        2  Zumajo, Norberto Palomares ★

  Grupo 5 (3 URIs, 7 artículos)
    Slug: eulalia maria subira
    Canónico: Subira, María Eulàlia
        1  Eulàlia Subirà, Mari

In [6]:
# Construir mapeo URI → URI canónica (combinando v1 + v2)
uri_to_canonical = {}
slug_to_canonical_uri = {}
slug_to_final_name = {}

# Primero: para cada cluster fuzzy, elegir URI canónica
for cluster in merge_clusters:
    # Elegir la URI con más artículos
    best_uri = max(cluster["uris"], key=lambda u: author_data[u]["n_articles"])
    canonical_name = author_data[best_uri]["name"]
    
    for uri in cluster["uris"]:
        uri_to_canonical[uri] = best_uri
    
    for slug in cluster["slugs"]:
        slug_to_canonical_uri[slug] = best_uri
        slug_to_final_name[slug] = canonical_name

# Para slugs no fusionados por fuzzy, usar su propia URI canónica
for slug, uris in slug_to_uris.items():
    if slug not in slug_to_canonical_uri:
        best_uri = max(uris, key=lambda u: author_data[u]["n_articles"])
        for uri in uris:
            uri_to_canonical[uri] = best_uri
        slug_to_canonical_uri[slug] = best_uri
        slug_to_final_name[slug] = author_data[best_uri]["name"]

total_mapped = len(uri_to_canonical)
total_canonical = len(set(uri_to_canonical.values()))
print(f"Mapeo construido: {total_mapped} URIs → {total_canonical} canónicas")
print(f"Verificación OK: no hay conflictos de mapeo")


Mapeo construido: 2794 URIs → 2407 canónicas
Verificación OK: no hay conflictos de mapeo


In [7]:
# Construir grafo dedup-v2 (desde original)
g_new = Graph()

g_new.bind("bibo", BIBO)
g_new.bind("dc", DC_)
g_new.bind("foaf", FOAF)
g_new.bind("skos", SKOS)
g_new.bind("idearq", IDEARQ)
g_new.bind("xsd", XSD)

for s, p, o in g:
    s_str = str(s)
    
    if s_str in uri_to_canonical:
        s = URIRef(uri_to_canonical[s_str])
    
    if p == DC_.creator and str(o) in uri_to_canonical:
        o = URIRef(uri_to_canonical[str(o)])
    
    g_new.add((s, p, o))

# Actualizar foaf:name
names_to_remove = list(g_new.triples((None, FOAF.name, None)))
for s, p, o in names_to_remove:
    g_new.remove((s, p, o))

for uri, d in author_data.items():
    canonical_uri = uri_to_canonical.get(uri, uri)
    slug = uri_to_slug[uri]
    name = slug_to_final_name.get(slug, d["name"])
    g_new.add((URIRef(canonical_uri), FOAF.name, Literal(name)))
    g_new.add((URIRef(canonical_uri), RDF.type, FOAF.Person))

new_authors = defaultdict(list)
for s, p, o in g_new.triples((None, FOAF.name, None)):
    new_authors[str(s)].append(str(o))

print(f"Triples en grafo dedup-v2: {len(g_new):,}")
print(f"Triples en grafo original: {len(g):,}")
print(f"Triples eliminados:        {len(g) - len(g_new):,}")
print(f"URIs de autor originales:  {len(author_data)}")
print(f"URIs de autor en v2:       {len(new_authors)}")
print(f"Nombres únicos en v2:      {len(set(n for names in new_authors.values() for n in names))}")


Triples en grafo dedup-v2: 16,357
Triples en grafo original: 17,306
Triples eliminados:        949
URIs de autor originales:  2794
URIs de autor en v2:       2407
Nombres únicos en v2:      2407


In [8]:
print(f"Guardando {OUTPUT_NT} ...")
g_new.serialize(destination=str(OUTPUT_NT), format="nt")
print(f"  Tamaño: {OUTPUT_NT.stat().st_size / 1024:.1f} KB")

print(f"\nGuardando {OUTPUT_TTL} ...")
g_new.serialize(destination=str(OUTPUT_TTL), format="turtle")
print(f"  Tamaño: {OUTPUT_TTL.stat().st_size / 1024:.1f} KB")

# Round-trip
g_check = Graph()
g_check.parse(str(OUTPUT_NT), format="nt")
assert len(g_check) == len(g_new), f"Mismatch: {len(g_check)} != {len(g_new)}"
print(f"\nRound-trip OK: {len(g_check):,} triples")

Guardando /home/raglinux/RAG/data/biblio-graph/idearq-graph-instances-dedup-v2.nt ...
  Tamaño: 2672.9 KB

Guardando /home/raglinux/RAG/data/biblio-graph/idearq-graph-instances-dedup-v2.ttl ...


/home/raglinux/env_rag/lib/python3.12/site-packages/rdflib/plugins/serializers/nt.py:41: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


  Tamaño: 1539.8 KB



Round-trip OK: 16,357 triples


In [9]:
print("Verificación: top autores con >=8 publicaciones\n")

q = """
PREFIX dc: <http://purl.org/dc/terms/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT ?name (COUNT(DISTINCT ?article) AS ?n) WHERE {
  ?article dc:creator ?author .
  ?author foaf:name ?name .
}
GROUP BY ?name
HAVING (COUNT(DISTINCT ?article) >= 8)
ORDER BY DESC(?n)
"""

print("Grafo v2 (dedup completo: slug + fuzzy):")
for r in g_new.query(q):
    print(f"  {r.n:>3}  {r.name}")

print("\n--- Grafo original ---")
for r in g.query(q):
    print(f"  {r.n:>3}  {r.name}")


Verificación: top autores con >=8 publicaciones

Grafo v2 (dedup completo: slug + fuzzy):


   16  Antonio M. Monge Soares
   15  Pablo Arias Cabal
   15  João Luís Cardoso
   14  R. E. M. HEDGES
   14  Pérez Jordà, Guillem
   13  Manuel R González Morales
   13  Francisco Javier Jover Maestre
   13  Gibaja, Juan F.
   12  Germán Delibes de Castro
   12  Antonio Faustino Carvalho
   12  C. BRONK RAMSEY
   12  Jesús Sesma Sesma
   12  Leonardo García Sanjuán
   12  Iñigo García-Martínez de Lagrán
   12  Marta Díaz-Zorita Bonilla
   11  Pena-Chocarro, Leonor
   11  Rafael María Martínez Sánchez
   11  Rafael Garrido Pena
   11  WILLIAM H. WALDREN
   11  M. Pilar Prieto Martínez
   11  Víctor M. Guerrero Ayuso
   10  R. A. HOUSLEY
   10  López Sáez, José Antonio
   10  Primitiva Bueno Ramírez
   10  G. J. VAN KLINKEN
   10  Alfonso Alday Ruiz
    9  Ángel ESPARZA ARROYO
    9  Lawrence Guy Straus
    9  María del Pilar Utrilla Miranda
    9  Pérez Díaz, Sebastián
    9  Carmen Rosa Olaria Puyoles
    9  Gonzalo Aranda Jiménez
    9  José Antonio MUJIKA-ALUSTIZA
    9  Oreto Garc

   15  Pablo Arias Cabal
   14  R. E. M. HEDGES
   14  Antonio M. Monge Soares
   14  António M Monge Soares
   14  António M. Monge Soares
   12  Germán Delibes de Castro
   12  C. BRONK RAMSEY
   12  C. Bronk Ramsey
   12  João Luís Cardoso
   12  João Luis Cardoso
   11  Víctor M. Guerrero Ayuso
   10  Alfonso Alday Ruiz
   10  G. J. VAN KLINKEN
   10  Gibaja, Juan F.
   10  Jesús Sesma Sesma
   10  WILLIAM H. WALDREN
   10  William H. Waldren
   10  Francisco Javier Jover Maestre
   10  R. A. HOUSLEY
    9  María del Pilar Utrilla Miranda
    9  José María Rodanés Vicente
    9  Leonardo García Sanjuán
    9  Carmen Rosa Olaria Puyoles
    9  Pérez Jordà, Guillem
    9  Pérez-Jordà, Guillem
    9  Pérez-Jordá, Guillem
    9  Antonio Faustino Carvalho
    9  António Faustino Carvalho
    9  Ángel ESPARZA ARROYO
    9  Ángel Esparza-Arroyo
    9  Ángel Esparza Arroyo
    9  Javier Velasco Vázquez
    9  Javier Velasco?Vázquez
    9  Javier Velasco-Vázquez
    8  Alfredo Mederos Martí

In [10]:
# Re-generar HTML con opción C: sin nodo "Sin journal"
from rdflib.namespace import FOAF as FOAF_NS
import networkx as nx
from pyvis.network import Network

TOP_N_AUTHORS = 30
MIN_ARTICLES_PER_AUTHOR = 2
OUTPUT_HTML = Path.cwd() / "explorar-coautorias-dedup-v2.html"

# SPARQL: top N autores
sparql_top = f"""
PREFIX dc: <http://purl.org/dc/terms/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT ?author ?authorName (COUNT(?article) AS ?n) WHERE {{
  ?article dc:creator ?author .
  ?author foaf:name ?authorName .
}}
GROUP BY ?author ?authorName
ORDER BY DESC(?n)
LIMIT {TOP_N_AUTHORS}
"""

top_authors = list(g_new.query(sparql_top))
print(f"Top {len(top_authors)} autores:")
for row in top_authors:
    print(f"  {row.authorName:45s}  {row.n:>3} pubs")

author_uris = {str(row.author) for row in top_authors}
author_uris_str = " ".join(f"<{u}>" for u in author_uris)

# SPARQL: artículos + journals
sparql_details = f"""
PREFIX dc: <http://purl.org/dc/terms/>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT ?authorUri ?authorName ?articleUri ?articleTitle ?journalName WHERE {{
  VALUES ?authorUri {{ {author_uris_str} }}
  ?articleUri dc:creator ?authorUri ;
              dc:title ?articleTitle .
  ?authorUri foaf:name ?authorName .
  OPTIONAL {{
    ?articleUri dc:isPartOf ?journal .
    ?journal dc:title ?journalName .
  }}
}}
ORDER BY ?authorName ?articleTitle
"""

details = list(g_new.query(sparql_details))
print(f"\n  {len(details)} filas recuperadas")

# Filtrar por min artículos
author_article_count = Counter(str(r.authorUri) for r in details)
valid_authors = {a for a, c in author_article_count.items() if c >= MIN_ARTICLES_PER_AUTHOR}
details = [r for r in details if str(r.authorUri) in valid_authors]
print(f"  Después de filtrar (min {MIN_ARTICLES_PER_AUTHOR}): {len(details)} filas")

# Construir grafo networkx
G = nx.Graph()

for row in details:
    author_name = str(row.authorName)
    article_title = str(row.articleTitle)
    journal_name = str(row.journalName) if row.journalName else None

    # Nodo autor
    if not G.has_node(author_name):
        G.add_node(author_name, type="author", label=author_name,
                   size=10, color="#4A90D9", title=f"Autor: {author_name}")

    # Nodo artículo
    short = article_title[:60] + "..." if len(article_title) > 60 else article_title
    if not G.has_node(article_title):
        G.add_node(article_title, type="article", label=short,
                   size=5, color="#7ED321", title=article_title)

    G.add_edge(author_name, article_title, color="#999", width=1)

    # OPCION C: solo crear nodo journal si existe
    if journal_name:
        journal_key = f"📚 {journal_name}"
        if not G.has_node(journal_key):
            G.add_node(journal_key, type="journal", label=journal_name,
                       size=8, color="#F5A623", title=f"Revista: {journal_name}")
        G.add_edge(article_title, journal_key, color="#B0D4F1", width=1)

# Ajustar tamaños
for node in G.nodes:
    deg = G.degree(node)
    ntype = G.nodes[node]["type"]
    if ntype == "author":
        G.nodes[node]["size"] = max(10, min(deg * 3, 40))
    elif ntype == "journal":
        G.nodes[node]["size"] = max(12, min(deg * 2, 35))
    elif ntype == "article":
        G.nodes[node]["size"] = max(4, min(deg * 2, 15))

print(f"\nGrafo: {G.number_of_nodes()} nodos, {G.number_of_edges()} aristas")
type_counts = Counter(G.nodes[n]["type"] for n in G.nodes)
for t, c in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f"  {t:12s}: {c}")

# Generar HTML
print(f"\nGenerando visualización...")
net = Network(
    height="900px", width="100%", directed=False, notebook=False,
    bgcolor="#1a1a2e", font_color="#ffffff", cdn_resources="remote"
)
net.from_nx(G)
net.toggle_physics(True)
net.set_options("""
{
  "physics": {
    "enabled": true,
    "solver": "forceAtlas2Based",
    "forceAtlas2Based": {
      "gravitationalConstant": -80,
      "centralGravity": 0.01,
      "springLength": 120,
      "springConstant": 0.05,
      "damping": 0.4
    },
    "stabilization": {"enabled": true, "iterations": 200, "fit": true}
  },
  "interaction": {"hover": true, "tooltipDelay": 100, "hideEdgesOnDrag": true},
  "edges": {"smooth": {"enabled": true, "type": "dynamic"}}
}
""")

net.save_graph(str(OUTPUT_HTML))
html_size = OUTPUT_HTML.stat().st_size / 1024
print(f"\n✅ HTML guardado: {OUTPUT_HTML}")
print(f"   Tamaño: {html_size:.1f} KB")
print(f"   Ábrelo con doble clic en el navegador.")

Top 30 autores:
  Antonio M. Monge Soares                         16 pubs
  Pablo Arias Cabal                               15 pubs
  João Luís Cardoso                               15 pubs
  R. E. M. HEDGES                                 14 pubs
  Pérez Jordà, Guillem                            14 pubs
  Manuel R González Morales                       13 pubs
  Francisco Javier Jover Maestre                  13 pubs
  Gibaja, Juan F.                                 13 pubs
  Germán Delibes de Castro                        12 pubs
  Antonio Faustino Carvalho                       12 pubs
  C. BRONK RAMSEY                                 12 pubs
  Jesús Sesma Sesma                               12 pubs
  Leonardo García Sanjuán                         12 pubs
  Iñigo García-Martínez de Lagrán                 12 pubs
  Marta Díaz-Zorita Bonilla                       12 pubs
  Pena-Chocarro, Leonor                           11 pubs
  Rafael María Martínez Sánchez                   11 pub

In [11]:
print("="*60)
print("  DEDUPLICACIÓN COMPLETA (slug exacto + fuzzy matching)")
print("="*60)
print(f"\n  Grafo original:     {len(g):,} triples, {len(author_data)} autores")
print(f"  Grafo dedup-v2:     {len(g_new):,} triples, {len(new_authors)} autores")
print(f"\n  Fusiones slug exacto: {v1_merged} URIs en {len(v1_clusters)} grupos")
print(f"  Fusiones fuzzy:       {sum(len(c['uris'])-1 for c in merge_clusters)} URIs en {len(merge_clusters)} grupos")
print(f"  Casos dudosos:        {len(doubtful_pairs)} (sim {THRESHOLD_REVIEW}-{THRESHOLD_MERGE})")
print(f"\n  Archivos:")
print(f"    {OUTPUT_NT}")
print(f"    {OUTPUT_TTL}")
print(f"    {OUTPUT_HTML}")


  DEDUPLICACIÓN COMPLETA (slug exacto + fuzzy matching)

  Grafo original:     17,306 triples, 2794 autores
  Grafo dedup-v2:     16,357 triples, 2407 autores

  Fusiones slug exacto: 317 URIs en 288 grupos
  Fusiones fuzzy:       94 URIs en 66 grupos
  Casos dudosos:        236 (sim 0.7-0.85)

  Archivos:
    /home/raglinux/RAG/data/biblio-graph/idearq-graph-instances-dedup-v2.nt
    /home/raglinux/RAG/data/biblio-graph/idearq-graph-instances-dedup-v2.ttl
    /home/raglinux/RAG/notebooks/graph/explorar-coautorias-dedup-v2.html
